In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import time
from scipy.sparse import csr_matrix

# Data

In [ ]:
interactions = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/interactions_train.csv")
interactions = interactions.rename(columns={'u': 'user_id', 'i': 'book_id', 't': 'timestamp'})
books = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/items.csv")

# Load classified books and merge
books_classified = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/books_classified.csv")
books = books.merge(books_classified[['i', 'book_type', 'discipline', 'topic', 'confidence']],
                    on='i', how='left')

user_id_map = {orig: new for new, orig in enumerate(interactions['user_id'].unique())}
book_id_map = {orig: new for new, orig in enumerate(interactions['book_id'].unique())}
user_id_inverse_map = {v: k for k, v in user_id_map.items()}
book_id_inverse_map = {v: k for k, v in book_id_map.items()}

interactions['user_id'] = interactions['user_id'].map(user_id_map)
interactions['book_id'] = interactions['book_id'].map(book_id_map)

n_users = interactions['user_id'].nunique()
n_items = interactions['book_id'].nunique()
print(f"Users: {n_users}, Items: {n_items}")
print(f"Books with categories: {books['book_type'].notna().sum()}")

Users: 7838, Items: 15109
Books with categories: 15291


# Functions with temporal weighting:

In [ ]:
def create_data_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    data_matrix[data["user_id"].values, data["book_id"].values] = 1
    return data_matrix

def create_weighted_matrix(data, n_users, n_items, decay=0.03):
    data_matrix = np.zeros((n_users, n_items))
    for user_id, user_data in data.groupby('user_id'):
        user_data = user_data.sort_values('timestamp')
        n = len(user_data)
        weights = np.array([(1 - decay) ** (n - 1 - i) for i in range(n)])
        if weights.max() > 0:
            weights = weights / weights.max()
        for i, (_, row) in enumerate(user_data.iterrows()):
            data_matrix[int(row['user_id']), int(row['book_id'])] = weights[i]
    return data_matrix

def user_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions) / (np.abs(similarity).sum(axis=1)[:, np.newaxis] + epsilon)
    return pred

def item_based_predict(interactions, similarity, epsilon=1e-9):
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T

def precision_recall_at_k(prediction, ground_truth, k=10):
    num_users = prediction.shape[0]
    precision_at_k, recall_at_k = 0, 0
    for user in range(num_users):
        top_k_items = np.argsort(prediction[user, :])[-k:]
        relevant = np.isin(top_k_items, np.where(ground_truth[user, :] == 1)[0]).sum()
        total = ground_truth[user, :].sum()
        precision_at_k += relevant / k
        recall_at_k += relevant / total if total > 0 else 0
    return precision_at_k / num_users, recall_at_k / num_users

def normalize_matrix(matrix):
    min_val = matrix.min()
    max_val = matrix.max()
    return (matrix - min_val) / (max_val - min_val + 1e-9)

def normalize_zscore(matrix):
    mean = matrix.mean()
    std = matrix.std()
    return (matrix - mean) / (std + 1e-9)

def create_count_weighted_matrix(data, n_users, n_items):
    """Weight interactions by how many times user took the book"""
    data_matrix = np.zeros((n_users, n_items))

    # Count interactions per user-book pair
    counts = data.groupby(['user_id', 'book_id']).size().reset_index(name='count')

    for _, row in counts.iterrows():
        # Log weight — 1 time=1.0, 2 times=1.58, 3 times=2.0
        data_matrix[int(row['user_id']), int(row['book_id'])] = np.log1p(row['count'])

    return data_matrix

print("Functions ready!")

Functions ready!


# TF-IDF

In [ ]:
books['Author'] = books['Author'].fillna('')
books['Subjects'] = books['Subjects'].fillna('')
books['Publisher'] = books['Publisher'].fillna('')
books['Title'] = books['Title'].fillna('')

books['v_author_x2'] = (books['Title'] + ' ' +
                         books['Author'] + ' ' + books['Author'] + ' ' +
                         books['Subjects'] + ' ' + books['Subjects'] + ' ' +
                         books['Publisher'])

tfidf_author_x2 = TfidfVectorizer(max_features=10000, stop_words=None,
                                    strip_accents='unicode', min_df=2)
tfidf_matrix_author_x2 = tfidf_author_x2.fit_transform(books['v_author_x2'])

mapping_author_x2 = {book_id_map[orig_id]: idx
                      for idx, orig_id in enumerate(books['i'])
                      if orig_id in book_id_map}
print(f"TF-IDF ready! Shape: {tfidf_matrix_author_x2.shape}")

TF-IDF ready! Shape: (15291, 10000)


# Idea 1: Pirson instead of cosine similarity

In [ ]:
import time
from scipy.stats import pearsonr

K_FOLDS = 5
scores_cosine = []
scores_pearson = []

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # Cosine similarity (current)
    user_sim_cos = cosine_similarity(train_matrix)
    user_pred_cos = user_based_predict(train_matrix, user_sim_cos)
    del user_sim_cos

    item_sim_cos = cosine_similarity(train_matrix.T)
    item_pred_cos = item_based_predict(train_matrix, item_sim_cos)
    del item_sim_cos

    # Pearson similarity
    user_sim_pear = np.corrcoef(train_matrix)
    user_sim_pear = np.nan_to_num(user_sim_pear)  # replace NaN with 0
    user_pred_pear = user_based_predict(train_matrix, user_sim_pear)
    del user_sim_pear

    item_sim_pear = np.corrcoef(train_matrix.T)
    item_sim_pear = np.nan_to_num(item_sim_pear)
    item_pred_pear = item_based_predict(train_matrix, item_sim_pear)
    del item_sim_pear

    # Content
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(content_pred)
    del content_pred

    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(book_popularity)

    # Evaluate cosine
    cf_cos = 0.45 * normalize_matrix(user_pred_cos) + 0.55 * normalize_matrix(item_pred_cos)
    hybrid_cos = 0.75 * cf_cos + 0.20 * content_norm + 0.05 * pop_norm
    precision_cos, _ = precision_recall_at_k(hybrid_cos, test_matrix, k=10)
    scores_cosine.append(precision_cos)
    print(f"  cosine  | Precision@10: {precision_cos:.4f}")
    del cf_cos, hybrid_cos

    # Evaluate pearson
    cf_pear = 0.45 * normalize_matrix(user_pred_pear) + 0.55 * normalize_matrix(item_pred_pear)
    hybrid_pear = 0.75 * cf_pear + 0.20 * content_norm + 0.05 * pop_norm
    precision_pear, _ = precision_recall_at_k(hybrid_pear, test_matrix, k=10)
    scores_pearson.append(precision_pear)
    print(f"  pearson | Precision@10: {precision_pear:.4f}")
    del cf_pear, hybrid_pear, user_pred_cos, item_pred_cos, user_pred_pear, item_pred_pear
    del content_norm, pop_norm

print("\n=== Average Precision@10 across 5 folds ===")
print(f"cosine  | Avg Precision@10: {np.mean(scores_cosine):.4f}")
print(f"pearson | Avg Precision@10: {np.mean(scores_pearson):.4f}")


── Fold 1/5 ──


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  cosine  | Precision@10: 0.0595
  pearson | Precision@10: 0.0564

── Fold 2/5 ──


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  cosine  | Precision@10: 0.0483
  pearson | Precision@10: 0.0449

── Fold 3/5 ──


...

  cosine  | Precision@10: 0.0593
  pearson | Precision@10: 0.0553

── Fold 4/5 ──


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  cosine  | Precision@10: 0.0506
  pearson | Precision@10: 0.0464

── Fold 5/5 ──


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


  cosine  | Precision@10: 0.0599
  pearson | Precision@10: 0.0562

=== Average Precision@10 across 5 folds ===
cosine  | Avg Precision@10: 0.0555
pearson | Avg Precision@10: 0.0519


# Idea 2: different neighbourhood size

In [ ]:
K_FOLDS = 5
top_n_values = [50, 100, 200, 500, n_users]  # n_users = все (текущий вариант)
scores_per_n = {n: [] for n in top_n_values}

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # Compute similarities once
    user_sim = cosine_similarity(train_matrix)
    item_sim = cosine_similarity(train_matrix.T)

    # Content
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(content_pred)
    del content_pred

    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(book_popularity)

    # Try different top-N
    for top_n in top_n_values:
        # Zero out similarities below top-N threshold
        user_sim_topn = user_sim.copy()
        for i in range(user_sim_topn.shape[0]):
            threshold = np.sort(user_sim_topn[i])[-min(top_n, len(user_sim_topn[i]))]
            user_sim_topn[i][user_sim_topn[i] < threshold] = 0

        item_sim_topn = item_sim.copy()
        for i in range(item_sim_topn.shape[0]):
            threshold = np.sort(item_sim_topn[i])[-min(top_n, len(item_sim_topn[i]))]
            item_sim_topn[i][item_sim_topn[i] < threshold] = 0

        user_pred = user_based_predict(train_matrix, user_sim_topn)
        item_pred = item_based_predict(train_matrix, item_sim_topn)
        del user_sim_topn, item_sim_topn

        cf_hybrid = 0.45 * normalize_matrix(user_pred) + 0.55 * normalize_matrix(item_pred)
        del user_pred, item_pred

        hybrid = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
        precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
        scores_per_n[top_n].append(precision)
        print(f"  top_n={top_n:5d} | Precision@10: {precision:.4f}")
        del cf_hybrid, hybrid

    del user_sim, item_sim, content_norm, pop_norm

print("\n=== Average Precision@10 across 5 folds ===")
best_n, best_score = None, 0
for n in top_n_values:
    avg = np.mean(scores_per_n[n])
    print(f"top_n={n:5d} | Avg Precision@10: {avg:.4f}")
    if avg > best_score:
        best_score = avg
        best_n = n

print(f"\n✓ Best top_n: {best_n} → Avg Precision@10: {best_score:.4f}")


── Fold 1/5 ──
  top_n=   50 | Precision@10: 0.0591
  top_n=  100 | Precision@10: 0.0594
  top_n=  200 | Precision@10: 0.0594
  top_n=  500 | Precision@10: 0.0595
  top_n= 7838 | Precision@10: 0.0595

── Fold 2/5 ──
  top_n=   50 | Precision@10: 0.0480
  top_n=  100 | Precision@10: 0.0483
  top_n=  200 | Precision@10: 0.0484
  top_n=  500 | Precision@10: 0.0483
  top_n= 7838 | Precision@10: 0.0483

── Fold 3/5 ──
  top_n=   50 | Precision@10: 0.0592
  top_n=  100 | Precision@10: 0.0592
  top_n=  200 | Precision@10: 0.0593
  top_n=  500 | Precision@10: 0.0593
  top_n= 7838 | Precision@10: 0.0593

── Fold 4/5 ──
  top_n=   50 | Precision@10: 0.0507
  top_n=  100 | Precision@10: 0.0507
  top_n=  200 | Precision@10: 0.0507
  top_n=  500 | Precision@10: 0.0506
  top_n= 7838 | Precision@10: 0.0506

── Fold 5/5 ──
  top_n=   50 | Precision@10: 0.0599
  top_n=  100 | Precision@10: 0.0599
  top_n=  200 | Precision@10: 0.0599
  top_n=  500 | Precision@10: 0.0599
  top_n= 7838 | Precision@10: 0.

# Idea 3: мы всегда нормализовали каждую матрицу отдельно перед смешиванием. А что если нормализовать финальную гибридную матрицу после смешивания?

In [ ]:
K_FOLDS = 5
approaches = {
    'current':     'normalize each then mix',
    'mix_then_norm': 'mix then normalize',
    'log_transform': 'log transform then normalize'
}
scores_per_approach = {a: [] for a in approaches}

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF predictions
    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    # Content
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()

    # ── Approach 1: current (normalize each then mix) ──
    cf_norm = 0.45 * normalize_matrix(user_pred) + 0.55 * normalize_matrix(item_pred)
    hybrid1 = 0.75 * cf_norm + 0.20 * normalize_matrix(content_pred) + 0.05 * normalize_matrix(book_popularity)
    precision1, _ = precision_recall_at_k(hybrid1, test_matrix, k=10)
    scores_per_approach['current'].append(precision1)
    print(f"  current          | Precision@10: {precision1:.4f}")
    del cf_norm, hybrid1

    # ── Approach 2: mix then normalize ──
    cf_raw = 0.45 * user_pred + 0.55 * item_pred
    hybrid2_raw = 0.75 * cf_raw + 0.20 * content_pred + 0.05 * book_popularity
    hybrid2 = normalize_matrix(hybrid2_raw)
    precision2, _ = precision_recall_at_k(hybrid2, test_matrix, k=10)
    scores_per_approach['mix_then_norm'].append(precision2)
    print(f"  mix_then_norm    | Precision@10: {precision2:.4f}")
    del cf_raw, hybrid2_raw, hybrid2

    # ── Approach 3: log transform then normalize ──
    user_pred_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
    item_pred_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
    content_pred_log = np.log1p(content_pred)
    pop_log = np.log1p(book_popularity)

    cf_log = 0.45 * normalize_matrix(user_pred_log) + 0.55 * normalize_matrix(item_pred_log)
    hybrid3 = 0.75 * cf_log + 0.20 * normalize_matrix(content_pred_log) + 0.05 * normalize_matrix(pop_log)
    precision3, _ = precision_recall_at_k(hybrid3, test_matrix, k=10)
    scores_per_approach['log_transform'].append(precision3)
    print(f"  log_transform    | Precision@10: {precision3:.4f}")
    del user_pred_log, item_pred_log, content_pred_log, pop_log, cf_log, hybrid3

    del user_pred, item_pred, content_pred, book_popularity

# Results
print("\n=== Average Precision@10 across 5 folds ===")
best_approach, best_score = None, 0
for a in approaches:
    avg = np.mean(scores_per_approach[a])
    print(f"{a:20s} | Avg Precision@10: {avg:.4f}")
    if avg > best_score:
        best_score = avg
        best_approach = a

print(f"\n✓ Best: {best_approach} → Avg Precision@10: {best_score:.4f}")


── Fold 1/5 ──
  current          | Precision@10: 0.0595
  mix_then_norm    | Precision@10: 0.0055
  log_transform    | Precision@10: 0.0596

── Fold 2/5 ──
  current          | Precision@10: 0.0483
  mix_then_norm    | Precision@10: 0.0030
  log_transform    | Precision@10: 0.0483

── Fold 3/5 ──
  current          | Precision@10: 0.0593
  mix_then_norm    | Precision@10: 0.0040
  log_transform    | Precision@10: 0.0594

── Fold 4/5 ──
  current          | Precision@10: 0.0506
  mix_then_norm    | Precision@10: 0.0028
  log_transform    | Precision@10: 0.0507

── Fold 5/5 ──
  current          | Precision@10: 0.0599
  mix_then_norm    | Precision@10: 0.0045
  log_transform    | Precision@10: 0.0600

=== Average Precision@10 across 5 folds ===
current              | Avg Precision@10: 0.0555
mix_then_norm        | Avg Precision@10: 0.0040
log_transform        | Avg Precision@10: 0.0556

✓ Best: log_transform → Avg Precision@10: 0.0556


# Now train on full data and save the file with recommendations:

In [ ]:
# Final submission with log transform
full_matrix = create_weighted_matrix(interactions, n_users, n_items, decay=0.03)

user_sim_full = cosine_similarity(full_matrix)
user_pred_full = user_based_predict(full_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_matrix.T)
item_pred_full = item_based_predict(full_matrix, item_sim_full)
del item_sim_full

# Log transform
user_pred_log = np.log1p(np.abs(user_pred_full)) * np.sign(user_pred_full)
item_pred_log = np.log1p(np.abs(item_pred_full)) * np.sign(item_pred_full)
del user_pred_full, item_pred_full

cf_hybrid = 0.45 * normalize_matrix(user_pred_log) + 0.55 * normalize_matrix(item_pred_log)
del user_pred_log, item_pred_log
print("CF hybrid ready!")

content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue
    rows = [mapping_author_x2[idx] for idx in read_indices
            if idx in mapping_author_x2]
    if len(rows) == 0:
        continue
    user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_log = np.log1p(content_pred)
content_norm = normalize_matrix(content_log)
del content_pred, content_log
print("Content ready!")

full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
pop_norm = normalize_matrix(np.log1p(book_popularity))

hybrid_final = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
del cf_hybrid, content_norm, pop_norm
print("Hybrid ready!")

K = 10
with open("submission_log.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_log.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

CF hybrid ready!
Content ready!
Hybrid ready!
user_id,recommendation
4456,13807 13805 9306 3333 8581 14120 9208 12689 13806 13804
142,1973 1970 1972 1959 1974 806 1568 1975 1962 1977
362,4248 4257 4249 4259 4256 4252 1418 2107 4258 4236
1809,8971 11317 8937 6338 5864 7496 5045 8653 9249 4824


# Idea 4: Weight of popularity - find the best one

In [ ]:
K_FOLDS = 5
pop_weights = [0.0, 0.03, 0.05, 0.08, 0.10, 0.15]
scores_per_pop = {w: [] for w in pop_weights}

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF with log transform
    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
    item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
    del user_pred, item_pred

    cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
    del user_log, item_log

    # Content with log transform
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(np.log1p(content_pred))
    del content_pred

    # Popularity with log transform
    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(np.log1p(book_popularity))

    # Try different popularity weights
    for pop_w in pop_weights:
        cf_w = 1 - 0.20 - pop_w
        if cf_w < 0:
            continue
        hybrid = cf_w * cf_hybrid + 0.20 * content_norm + pop_w * pop_norm
        precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
        scores_per_pop[pop_w].append(precision)
        print(f"  pop_w={pop_w:.2f} cf_w={cf_w:.2f} | Precision@10: {precision:.4f}")
        del hybrid

    del cf_hybrid, content_norm, pop_norm

# Results
print("\n=== Average Precision@10 across 5 folds ===")
best_pop_w, best_score = None, 0
for w in pop_weights:
    if scores_per_pop[w]:
        avg = np.mean(scores_per_pop[w])
        print(f"pop_w={w:.2f} | Avg Precision@10: {avg:.4f}")
        if avg > best_score:
            best_score = avg
            best_pop_w = w

print(f"\n✓ Best pop_w: {best_pop_w} → Avg Precision@10: {best_score:.4f}")


── Fold 1/5 ──
  pop_w=0.00 cf_w=0.80 | Precision@10: 0.0592
  pop_w=0.03 cf_w=0.77 | Precision@10: 0.0595
  pop_w=0.05 cf_w=0.75 | Precision@10: 0.0596
  pop_w=0.08 cf_w=0.72 | Precision@10: 0.0597
  pop_w=0.10 cf_w=0.70 | Precision@10: 0.0599
  pop_w=0.15 cf_w=0.65 | Precision@10: 0.0594

── Fold 2/5 ──
  pop_w=0.00 cf_w=0.80 | Precision@10: 0.0483
  pop_w=0.03 cf_w=0.77 | Precision@10: 0.0483
  pop_w=0.05 cf_w=0.75 | Precision@10: 0.0483
  pop_w=0.08 cf_w=0.72 | Precision@10: 0.0483
  pop_w=0.10 cf_w=0.70 | Precision@10: 0.0482
  pop_w=0.15 cf_w=0.65 | Precision@10: 0.0476

── Fold 3/5 ──
  pop_w=0.00 cf_w=0.80 | Precision@10: 0.0589
  pop_w=0.03 cf_w=0.77 | Precision@10: 0.0593
  pop_w=0.05 cf_w=0.75 | Precision@10: 0.0594
  pop_w=0.08 cf_w=0.72 | Precision@10: 0.0594
  pop_w=0.10 cf_w=0.70 | Precision@10: 0.0592
  pop_w=0.15 cf_w=0.65 | Precision@10: 0.0587

── Fold 4/5 ──
  pop_w=0.00 cf_w=0.80 | Precision@10: 0.0505
  pop_w=0.03 cf_w=0.77 | Precision@10: 0.0506
  pop_w=0.05 cf_

# Now train on full data (with 0.1 weight for popularity, 0.7 CF, 0,2 content and log transform) and save the file with recommendations:

# **NO IMPROVEMENT ON REAL DATA**




In [ ]:
full_matrix = create_weighted_matrix(interactions, n_users, n_items, decay=0.03)

user_sim_full = cosine_similarity(full_matrix)
user_pred_full = user_based_predict(full_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_matrix.T)
item_pred_full = item_based_predict(full_matrix, item_sim_full)
del item_sim_full

user_log = np.log1p(np.abs(user_pred_full)) * np.sign(user_pred_full)
item_log = np.log1p(np.abs(item_pred_full)) * np.sign(item_pred_full)
del user_pred_full, item_pred_full

cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
del user_log, item_log
print("CF ready!")

content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue
    rows = [mapping_author_x2[idx] for idx in read_indices
            if idx in mapping_author_x2]
    if len(rows) == 0:
        continue
    user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_norm = normalize_matrix(np.log1p(content_pred))
del content_pred
print("Content ready!")

full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
pop_norm = normalize_matrix(np.log1p(book_popularity))

hybrid_final = 0.70 * cf_hybrid + 0.20 * content_norm + 0.10 * pop_norm
del cf_hybrid, content_norm, pop_norm
print("Hybrid ready!")

K = 10
with open("submission_log_pop10.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_log_pop10.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

KeyboardInterrupt: 

# Idea 5: adaptive weight dependind on how many books person has read (NO IMPROVEMENT)

In [ ]:
K_FOLDS = 5
scores_adaptive = []
scores_current = []

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

# Compute activity per user on full data
user_activity = interactions.groupby('user_id').size()

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF
    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
    item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
    del user_pred, item_pred

    user_norm = normalize_matrix(user_log)
    item_norm = normalize_matrix(item_log)
    del user_log, item_log

    cf_hybrid = 0.45 * user_norm + 0.55 * item_norm
    del user_norm, item_norm

    # Content
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(np.log1p(content_pred))
    del content_pred

    # Popularity
    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(np.log1p(book_popularity))

    # ── Current model (fixed weights) ──
    hybrid_current = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
    precision_cur, _ = precision_recall_at_k(hybrid_current, test_matrix, k=10)
    scores_current.append(precision_cur)
    print(f"  current   | Precision@10: {precision_cur:.4f}")
    del hybrid_current

    # ── Adaptive model (per-user weights based on activity) ──
    hybrid_adaptive = np.zeros((n_users, n_items))

    for user_idx in range(n_users):
        original_user_id = user_id_inverse_map[user_idx]
        activity = user_activity.get(original_user_id, 0)

        if activity <= 3:
            # Cold user — trust popularity more
            cf_w, content_w, pop_w = 0.50, 0.20, 0.30
        elif activity <= 6:
            # Medium user
            cf_w, content_w, pop_w = 0.65, 0.20, 0.15
        else:
            # Active user — trust CF more
            cf_w, content_w, pop_w = 0.80, 0.15, 0.05

        hybrid_adaptive[user_idx, :] = (
            cf_w * cf_hybrid[user_idx, :] +
            content_w * content_norm[user_idx, :] +
            pop_w * pop_norm
        )

    precision_ada, _ = precision_recall_at_k(hybrid_adaptive, test_matrix, k=10)
    scores_adaptive.append(precision_ada)
    print(f"  adaptive  | Precision@10: {precision_ada:.4f}")
    del hybrid_adaptive, cf_hybrid, content_norm, pop_norm

# Results
print("\n=== Average Precision@10 across 5 folds ===")
print(f"current  | Avg Precision@10: {np.mean(scores_current):.4f}")
print(f"adaptive | Avg Precision@10: {np.mean(scores_adaptive):.4f}")


── Fold 1/5 ──
  current   | Precision@10: 0.0596
  adaptive  | Precision@10: 0.0581

── Fold 2/5 ──
  current   | Precision@10: 0.0483
  adaptive  | Precision@10: 0.0469

── Fold 3/5 ──
  current   | Precision@10: 0.0594
  adaptive  | Precision@10: 0.0569

── Fold 4/5 ──


KeyboardInterrupt: 

# Idea 6: use popularity of recent books

In [ ]:
K_FOLDS = 5

# Test different combinations of overall_pop and recent_pop
# Format: (overall_w, recent_w) — CF gets 1 - 0.20 - overall_w - recent_w
combos = [
    (0.05, 0.00),  # baseline — no recent
    (0.03, 0.02),  # 5% total, split
    (0.05, 0.05),  # 10% total, equal split
    (0.03, 0.07),  # 10% total, more recent
    (0.02, 0.08),  # 10% total, mostly recent
    (0.05, 0.10),  # 15% total
    (0.00, 0.10),  # only recent popularity
]

scores_per_combo = {c: [] for c in combos}

# Build recent popularity vector
interactions['date'] = pd.to_datetime(interactions['timestamp'], unit='s')
last_date = interactions['date'].max()
recent = interactions[interactions['date'] >= last_date - pd.Timedelta(days=30)]
recent_popular = recent.groupby('book_id').size()

recent_pop = np.zeros(n_items)
for orig_id, count in recent_popular.items():
    if orig_id in book_id_map:
        recent_pop[book_id_map[orig_id]] = count

recent_pop_norm = normalize_matrix(np.log1p(recent_pop))

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF
    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
    item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
    del user_pred, item_pred

    cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
    del user_log, item_log

    # Content
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(np.log1p(content_pred))
    del content_pred

    # Overall popularity from train
    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    overall_pop_norm = normalize_matrix(np.log1p(book_popularity))

    # Evaluate each combo
    for (overall_w, recent_w) in combos:
        cf_w = 1 - 0.20 - overall_w - recent_w
        if cf_w < 0:
            print(f"  overall={overall_w} recent={recent_w} | skipped")
            continue
        hybrid = (cf_w * cf_hybrid + 0.20 * content_norm +
                  overall_w * overall_pop_norm + recent_w * recent_pop_norm)
        precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
        scores_per_combo[(overall_w, recent_w)].append(precision)
        print(f"  overall={overall_w:.2f} recent={recent_w:.2f} cf={cf_w:.2f} | Precision@10: {precision:.4f}")
        del hybrid

    del cf_hybrid, content_norm, overall_pop_norm

# Results
print("\n=== Average Precision@10 across 5 folds ===")
best_combo, best_score = None, 0
for combo in combos:
    if scores_per_combo[combo]:
        avg = np.mean(scores_per_combo[combo])
        print(f"overall={combo[0]:.2f} recent={combo[1]:.2f} | Avg Precision@10: {avg:.4f}")
        if avg > best_score:
            best_score = avg
            best_combo = combo

print(f"\n✓ Best: overall={best_combo[0]} recent={best_combo[1]} → Avg Precision@10: {best_score:.4f}")


── Fold 1/5 ──
  overall=0.05 recent=0.00 cf=0.75 | Precision@10: 0.0596
  overall=0.03 recent=0.02 cf=0.75 | Precision@10: 0.0594
  overall=0.05 recent=0.05 cf=0.70 | Precision@10: 0.0596
  overall=0.03 recent=0.07 cf=0.70 | Precision@10: 0.0594
  overall=0.02 recent=0.08 cf=0.70 | Precision@10: 0.0593
  overall=0.05 recent=0.10 cf=0.65 | Precision@10: 0.0592
  overall=0.00 recent=0.10 cf=0.70 | Precision@10: 0.0589

── Fold 2/5 ──
  overall=0.05 recent=0.00 cf=0.75 | Precision@10: 0.0483
  overall=0.03 recent=0.02 cf=0.75 | Precision@10: 0.0483
  overall=0.05 recent=0.05 cf=0.70 | Precision@10: 0.0484
  overall=0.03 recent=0.07 cf=0.70 | Precision@10: 0.0484
  overall=0.02 recent=0.08 cf=0.70 | Precision@10: 0.0483
  overall=0.05 recent=0.10 cf=0.65 | Precision@10: 0.0479
  overall=0.00 recent=0.10 cf=0.70 | Precision@10: 0.0480

── Fold 3/5 ──
  overall=0.05 recent=0.00 cf=0.75 | Precision@10: 0.0594
  overall=0.03 recent=0.02 cf=0.75 | Precision@10: 0.0592
  overall=0.05 recent=0.

# Train on full data with 0.05 popularity and 0.05 recent popularity

In [ ]:
# ── Final model: CF + content + overall_pop + recent_pop + log transform ──

# Build recent popularity vector
interactions['date'] = pd.to_datetime(interactions['timestamp'], unit='s')
last_date = interactions['date'].max()
recent = interactions[interactions['date'] >= last_date - pd.Timedelta(days=30)]
recent_popular = recent.groupby('book_id').size()

recent_pop = np.zeros(n_items)
for orig_id, count in recent_popular.items():
    if orig_id in book_id_map:
        recent_pop[book_id_map[orig_id]] = count

recent_pop_norm = normalize_matrix(np.log1p(recent_pop))
print("Recent popularity ready!")

# Full weighted matrix
full_matrix = create_weighted_matrix(interactions, n_users, n_items, decay=0.03)

# CF
user_sim_full = cosine_similarity(full_matrix)
user_pred_full = user_based_predict(full_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_matrix.T)
item_pred_full = item_based_predict(full_matrix, item_sim_full)
del item_sim_full

user_log = np.log1p(np.abs(user_pred_full)) * np.sign(user_pred_full)
item_log = np.log1p(np.abs(item_pred_full)) * np.sign(item_pred_full)
del user_pred_full, item_pred_full

cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
del user_log, item_log
print("CF ready!")

# Content
content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue
    rows = [mapping_author_x2[idx] for idx in read_indices
            if idx in mapping_author_x2]
    if len(rows) == 0:
        continue
    user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_norm = normalize_matrix(np.log1p(content_pred))
del content_pred
print("Content ready!")

# Overall popularity
full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
overall_pop_norm = normalize_matrix(np.log1p(book_popularity))

# Final hybrid
# CF=0.70, content=0.20, overall_pop=0.05, recent_pop=0.05
hybrid_final = (0.70 * cf_hybrid + 0.20 * content_norm +
                0.05 * overall_pop_norm + 0.05 * recent_pop_norm)
del cf_hybrid, content_norm, overall_pop_norm, recent_pop_norm
print("Hybrid ready!")

# Save submission
K = 10
with open("submission_recent_pop.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_recent_pop.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

Recent popularity ready!
CF ready!
Content ready!
Hybrid ready!
user_id,recommendation
4456,13807 13805 9306 3333 8581 14120 9208 12689 13806 13804
142,1973 1970 1972 1959 1974 806 1975 1977 1568 1962
362,4248 4259 4249 4257 1418 4256 4252 4236 2107 4258
1809,8971 11317 8937 6338 5864 7496 5045 168 8653 9249


# Idea 7: попробуем веса по количеству взятий

In [ ]:
K_FOLDS = 5

approaches = ['temporal', 'count', 'temporal_x_count']
scores_per_approach = {a: [] for a in approaches}

# Count-based matrix function
def create_count_weighted_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    counts = data.groupby(['user_id', 'book_id']).size().reset_index(name='count')
    for _, row in counts.iterrows():
        data_matrix[int(row['user_id']), int(row['book_id'])] = np.log1p(row['count'])
    return data_matrix

# Temporal x Count matrix function
def create_temporal_count_matrix(data, n_users, n_items, decay=0.03):
    data_matrix = np.zeros((n_users, n_items))
    counts = data.groupby(['user_id', 'book_id']).size().reset_index(name='count')

    for user_id, user_data in data.groupby('user_id'):
        user_data = user_data.sort_values('timestamp')
        n = len(user_data)
        weights = np.array([(1 - decay) ** (n - 1 - i) for i in range(n)])
        if weights.max() > 0:
            weights = weights / weights.max()
        for i, (_, row) in enumerate(user_data.iterrows()):
            book_id = int(row['book_id'])
            # Get count for this user-book pair
            count = counts[(counts['user_id'] == user_id) &
                          (counts['book_id'] == book_id)]['count'].values
            count = count[0] if len(count) > 0 else 1
            # Combine temporal weight with count weight
            data_matrix[int(row['user_id']), book_id] = weights[i] * np.log1p(count)

    return data_matrix

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # Build three different matrices
    train_temporal = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)
    train_count = create_count_weighted_matrix(train_data, n_users, n_items)
    train_temporal_count = create_temporal_count_matrix(train_data, n_users, n_items, decay=0.03)

    # Content (same for all)
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(np.log1p(content_pred))
    del content_pred

    for approach, train_matrix in [('temporal', train_temporal),
                                    ('count', train_count),
                                    ('temporal_x_count', train_temporal_count)]:
        # CF
        user_sim = cosine_similarity(train_matrix)
        user_pred = user_based_predict(train_matrix, user_sim)
        del user_sim

        item_sim = cosine_similarity(train_matrix.T)
        item_pred = item_based_predict(train_matrix, item_sim)
        del item_sim

        user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
        item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
        del user_pred, item_pred

        cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
        del user_log, item_log

        book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
        pop_norm = normalize_matrix(np.log1p(book_popularity))

        hybrid = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
        precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
        scores_per_approach[approach].append(precision)
        print(f"  {approach:20s} | Precision@10: {precision:.4f}")
        del cf_hybrid, pop_norm, hybrid

    del train_temporal, train_count, train_temporal_count, content_norm

# Results
print("\n=== Average Precision@10 across 5 folds ===")
best_approach, best_score = None, 0
for a in approaches:
    avg = np.mean(scores_per_approach[a])
    print(f"{a:20s} | Avg Precision@10: {avg:.4f}")
    if avg > best_score:
        best_score = avg
        best_approach = a

print(f"\n✓ Best: {best_approach} → Avg Precision@10: {best_score:.4f}")


── Fold 1/5 ──
  temporal             | Precision@10: 0.0596
  count                | Precision@10: 0.0601
  temporal_x_count     | Precision@10: 0.0596

── Fold 2/5 ──
  temporal             | Precision@10: 0.0483
  count                | Precision@10: 0.0490
  temporal_x_count     | Precision@10: 0.0483

── Fold 3/5 ──
  temporal             | Precision@10: 0.0594
  count                | Precision@10: 0.0593
  temporal_x_count     | Precision@10: 0.0591

── Fold 4/5 ──
  temporal             | Precision@10: 0.0507
  count                | Precision@10: 0.0499
  temporal_x_count     | Precision@10: 0.0505

── Fold 5/5 ──
  temporal             | Precision@10: 0.0600
  count                | Precision@10: 0.0593
  temporal_x_count     | Precision@10: 0.0596

=== Average Precision@10 across 5 folds ===
temporal             | Avg Precision@10: 0.0556
count                | Avg Precision@10: 0.0555
temporal_x_count     | Avg Precision@10: 0.0554

✓ Best: temporal → Avg Precision@10: 0.0

# Idea 8: Graph

In [ ]:
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import norm

# ── Graph-based: Random Walk with Restart ─────────────────────────────────

# Step 1: Build bipartite graph (users + items as nodes)
# Rows 0..n_users-1 = users
# Rows n_users..n_users+n_items-1 = items

full_binary = create_data_matrix(interactions, n_users, n_items)
sparse_interactions = csr_matrix(full_binary)

# Build adjacency matrix of bipartite graph
# [0,        R ]
# [R.T,      0 ]
# where R = user-item interaction matrix

from scipy.sparse import bmat

R = sparse_interactions  # n_users × n_items
zero_users = csr_matrix((n_users, n_users))
zero_items = csr_matrix((n_items, n_items))

adjacency = bmat([[zero_users, R],
                  [R.T, zero_items]], format='csr')

print(f"Adjacency matrix shape: {adjacency.shape}")
print(f"Non-zero elements: {adjacency.nnz}")

# Step 2: Normalize adjacency matrix (row-wise)
row_sums = np.array(adjacency.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1  # avoid division by zero
D_inv = diags(1.0 / row_sums)
transition = D_inv.dot(adjacency)  # row-stochastic matrix

print("Transition matrix ready!")

# Step 3: Random Walk with Restart (RWR)
# For each user: start at user node, walk with restart probability alpha
# Score of each item = probability of visiting that item

alpha = 0.85  # restart probability (typical value)
n_iterations = 20

# Compute RWR scores for all users at once
# Initialize: each user starts at their own node
n_total = n_users + n_items

# User preference vectors (personalized)
# For each user u: p[u] = 1, rest = 0
scores_matrix = np.zeros((n_users, n_items))

print("Computing RWR scores...")
batch_size = 100  # process users in batches to save RAM

for batch_start in range(0, n_users, batch_size):
    batch_end = min(batch_start + batch_size, n_users)
    batch_size_actual = batch_end - batch_start

    # Initialize personalized vectors for this batch
    p = np.zeros((n_total, batch_size_actual))
    for i, user_idx in enumerate(range(batch_start, batch_end)):
        p[user_idx, i] = 1.0

    # RWR iterations
    r = p.copy()
    for _ in range(n_iterations):
        r = alpha * transition.dot(r) + (1 - alpha) * p

    # Extract item scores (rows n_users to n_users+n_items)
    scores_matrix[batch_start:batch_end, :] = r[n_users:, :].T

    if batch_start % 1000 == 0:
        print(f"  Processed {batch_start}/{n_users} users...")

print("RWR scores ready!")
print(f"Sample scores user 0: {scores_matrix[0, :5]}")

Adjacency matrix shape: (22947, 22947)
Non-zero elements: 128006
Transition matrix ready!
Computing RWR scores...
  Processed 0/7838 users...
  Processed 1000/7838 users...
  Processed 2000/7838 users...
  Processed 3000/7838 users...
  Processed 4000/7838 users...
  Processed 5000/7838 users...
  Processed 6000/7838 users...
  Processed 7000/7838 users...
RWR scores ready!
Sample scores user 0: [2.28081349e-02 2.15219065e-05 1.91091610e-05 2.28563927e-05
 1.99300414e-06]


In [ ]:
from scipy.sparse import csr_matrix, diags, bmat
import time

K_FOLDS = 5

# Different alpha values for RWR (restart probability)
alphas = [0.5, 0.7, 0.85, 0.95]
# Different ways to combine with current model
graph_weights = [0.0, 0.2, 0.5, 0.8, 1.0]

scores_results = {}  # (alpha, graph_w) → list of precisions

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # ── Build current best model ───────────────────────────────────────────
    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
    item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
    del user_pred, item_pred

    cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
    del user_log, item_log

    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(np.log1p(content_pred))
    del content_pred

    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(np.log1p(book_popularity))

    current_model = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
    del cf_hybrid, content_norm, pop_norm
    current_norm = normalize_matrix(current_model)
    del current_model

    # ── Build graph from train data ────────────────────────────────────────
    train_binary = create_data_matrix(train_data, n_users, n_items)
    R = csr_matrix(train_binary)
    del train_binary

    zero_users = csr_matrix((n_users, n_users))
    zero_items = csr_matrix((n_items, n_items))
    adjacency = bmat([[zero_users, R], [R.T, zero_items]], format='csr')
    del R, zero_users, zero_items

    row_sums = np.array(adjacency.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1
    D_inv = diags(1.0 / row_sums)
    transition = D_inv.dot(adjacency)
    del adjacency, D_inv

    # ── Try different alpha values ─────────────────────────────────────────
    for alpha in alphas:
        print(f"\n  Computing RWR alpha={alpha}...")
        start = time.time()

        n_total = n_users + n_items
        graph_scores = np.zeros((n_users, n_items))
        batch_size = 200

        for batch_start in range(0, n_users, batch_size):
            batch_end = min(batch_start + batch_size, n_users)
            batch_size_actual = batch_end - batch_start

            p = np.zeros((n_total, batch_size_actual))
            for i, user_idx in enumerate(range(batch_start, batch_end)):
                p[user_idx, i] = 1.0

            r = p.copy()
            for _ in range(15):  # 15 iterations enough
                r = alpha * transition.dot(r) + (1 - alpha) * p

            graph_scores[batch_start:batch_end, :] = r[n_users:, :].T

        graph_norm = normalize_matrix(graph_scores)
        del graph_scores
        print(f"  RWR done in {time.time()-start:.1f}s")

        # Try different graph weights
        for graph_w in graph_weights:
            key = (alpha, graph_w)
            if key not in scores_results:
                scores_results[key] = []

            if graph_w == 0.0:
                hybrid = current_norm
            elif graph_w == 1.0:
                hybrid = graph_norm
            else:
                hybrid = (1 - graph_w) * current_norm + graph_w * graph_norm

            precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
            scores_results[key].append(precision)
            print(f"    alpha={alpha} graph_w={graph_w:.1f} | Precision@10: {precision:.4f}")

        del graph_norm

    del transition, current_norm

# ── Results ────────────────────────────────────────────────────────────────
print("\n=== Average Precision@10 across 5 folds ===")
best_key, best_score = None, 0
for (alpha, graph_w), scores in sorted(scores_results.items()):
    avg = np.mean(scores)
    print(f"alpha={alpha} graph_w={graph_w:.1f} | Avg Precision@10: {avg:.4f}")
    if avg > best_score:
        best_score = avg
        best_key = (alpha, graph_w)

print(f"\n✓ Best: alpha={best_key[0]} graph_w={best_key[1]} → Avg Precision@10: {best_score:.4f}")


── Fold 1/5 ──

  Computing RWR alpha=0.5...
  RWR done in 59.8s
    alpha=0.5 graph_w=0.0 | Precision@10: 0.0596
    alpha=0.5 graph_w=0.2 | Precision@10: 0.0598
    alpha=0.5 graph_w=0.5 | Precision@10: 0.0598
    alpha=0.5 graph_w=0.8 | Precision@10: 0.0592
    alpha=0.5 graph_w=1.0 | Precision@10: 0.0547

  Computing RWR alpha=0.7...
  RWR done in 58.9s
    alpha=0.7 graph_w=0.0 | Precision@10: 0.0596
    alpha=0.7 graph_w=0.2 | Precision@10: 0.0599
    alpha=0.7 graph_w=0.5 | Precision@10: 0.0598
    alpha=0.7 graph_w=0.8 | Precision@10: 0.0593
    alpha=0.7 graph_w=1.0 | Precision@10: 0.0546

  Computing RWR alpha=0.85...
  RWR done in 61.7s
    alpha=0.85 graph_w=0.0 | Precision@10: 0.0596
    alpha=0.85 graph_w=0.2 | Precision@10: 0.0598
    alpha=0.85 graph_w=0.5 | Precision@10: 0.0596
    alpha=0.85 graph_w=0.8 | Precision@10: 0.0593
    alpha=0.85 graph_w=1.0 | Precision@10: 0.0542

  Computing RWR alpha=0.95...
  RWR done in 58.9s
    alpha=0.95 graph_w=0.0 | Precision@10:

# Train on the full data set with the best graph weight (0.2)

In [ ]:
from scipy.sparse import csr_matrix, diags, bmat

# ── Final model with Graph RWR ─────────────────────────────────────────────

# Step 1: Build current best model
full_matrix = create_weighted_matrix(interactions, n_users, n_items, decay=0.03)

user_sim_full = cosine_similarity(full_matrix)
user_pred_full = user_based_predict(full_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_matrix.T)
item_pred_full = item_based_predict(full_matrix, item_sim_full)
del item_sim_full

user_log = np.log1p(np.abs(user_pred_full)) * np.sign(user_pred_full)
item_log = np.log1p(np.abs(item_pred_full)) * np.sign(item_pred_full)
del user_pred_full, item_pred_full

cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
del user_log, item_log

content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue
    rows = [mapping_author_x2[idx] for idx in read_indices
            if idx in mapping_author_x2]
    if len(rows) == 0:
        continue
    user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_norm = normalize_matrix(np.log1p(content_pred))
del content_pred

full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
pop_norm = normalize_matrix(np.log1p(book_popularity))

current_model = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
del cf_hybrid, content_norm, pop_norm
current_norm = normalize_matrix(current_model)
del current_model
print("Current model ready!")

# Step 2: Build graph RWR
R = csr_matrix(full_binary)
zero_users = csr_matrix((n_users, n_users))
zero_items = csr_matrix((n_items, n_items))
adjacency = bmat([[zero_users, R], [R.T, zero_items]], format='csr')
del R, zero_users, zero_items

row_sums = np.array(adjacency.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1
D_inv = diags(1.0 / row_sums)
transition = D_inv.dot(adjacency)
del adjacency, D_inv
print("Graph ready!")

# RWR with alpha=0.7
alpha = 0.7
n_total = n_users + n_items
graph_scores = np.zeros((n_users, n_items))
batch_size = 200

for batch_start in range(0, n_users, batch_size):
    batch_end = min(batch_start + batch_size, n_users)
    batch_size_actual = batch_end - batch_start

    p = np.zeros((n_total, batch_size_actual))
    for i, user_idx in enumerate(range(batch_start, batch_end)):
        p[user_idx, i] = 1.0

    r = p.copy()
    for _ in range(15):
        r = alpha * transition.dot(r) + (1 - alpha) * p

    graph_scores[batch_start:batch_end, :] = r[n_users:, :].T

    if batch_start % 1000 == 0:
        print(f"  RWR: {batch_start}/{n_users} users...")

graph_norm = normalize_matrix(graph_scores)
del graph_scores, transition
print("RWR ready!")

# Step 3: Final hybrid
hybrid_final = 0.8 * current_norm + 0.2 * graph_norm
del current_norm, graph_norm
print("Hybrid ready!")

# Step 4: Save submission
K = 10
with open("submission_graph.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_graph.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

Current model ready!
Graph ready!
  RWR: 0/7838 users...
  RWR: 1000/7838 users...
  RWR: 2000/7838 users...
  RWR: 3000/7838 users...
  RWR: 4000/7838 users...
  RWR: 5000/7838 users...
  RWR: 6000/7838 users...
  RWR: 7000/7838 users...
RWR ready!
Hybrid ready!
user_id,recommendation
4456,13807 13805 3333 9306 9208 8581 14120 12689 13806 13804
142,1973 1970 1959 1972 1974 1962 1975 806 600 1977
362,4248 4258 4257 4259 4252 2107 4256 1418 4249 4246
1809,8971 6338 11317 8937 5864 7496 5045 8653 9249 4824


# Idea 9: weighted TF-IDF by counts

In [ ]:
K_FOLDS = 5
scores_standard = []
scores_count_weighted = []

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

# Precompute interaction counts
interaction_counts = interactions.groupby(['user_id', 'book_id']).size().reset_index(name='count')
count_dict = {(row['user_id'], row['book_id']): row['count']
              for _, row in interaction_counts.iterrows()}

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF
    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
    item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
    del user_pred, item_pred

    cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
    del user_log, item_log

    # Popularity
    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(np.log1p(book_popularity))

    # ── Standard TF-IDF profile (current) ─────────────────────────────────
    content_standard = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue
        # Simple average — all books equal weight
        user_profile = np.asarray(tfidf_matrix_author_x2[rows].mean(axis=0))
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_standard[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm_std = normalize_matrix(np.log1p(content_standard))
    del content_standard

    hybrid_std = 0.75 * cf_hybrid + 0.20 * content_norm_std + 0.05 * pop_norm
    precision_std, _ = precision_recall_at_k(hybrid_std, test_matrix, k=10)
    scores_standard.append(precision_std)
    print(f"  standard TF-IDF  | Precision@10: {precision_std:.4f}")
    del hybrid_std, content_norm_std

    # ── Count-weighted TF-IDF profile ─────────────────────────────────────
    content_count = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        rows = [mapping_author_x2[idx] for idx in train_books
                if idx in mapping_author_x2]
        if len(rows) == 0:
            continue

        # Get count weights for each book
        weights = []
        valid_rows = []
        for book_idx in train_books:
            if book_idx in mapping_author_x2:
                count = count_dict.get((user_idx, book_idx), 1)
                weights.append(np.log1p(count))
                valid_rows.append(mapping_author_x2[book_idx])

        if len(valid_rows) == 0:
            continue

        weights = np.array(weights)
        weights = weights / weights.sum()  # normalize weights

        # Weighted average TF-IDF profile
        user_profile = np.asarray(
            np.average(tfidf_matrix_author_x2[valid_rows].toarray(),
                      weights=weights, axis=0)
        ).reshape(1, -1)

        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_count[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm_count = normalize_matrix(np.log1p(content_count))
    del content_count

    hybrid_count = 0.75 * cf_hybrid + 0.20 * content_norm_count + 0.05 * pop_norm
    precision_count, _ = precision_recall_at_k(hybrid_count, test_matrix, k=10)
    scores_count_weighted.append(precision_count)
    print(f"  count TF-IDF     | Precision@10: {precision_count:.4f}")
    del hybrid_count, content_norm_count, cf_hybrid, pop_norm

# Results
print("\n=== Average Precision@10 across 5 folds ===")
print(f"standard TF-IDF  | Avg Precision@10: {np.mean(scores_standard):.4f}")
print(f"count TF-IDF     | Avg Precision@10: {np.mean(scores_count_weighted):.4f}")


── Fold 1/5 ──
  standard TF-IDF  | Precision@10: 0.0596
  count TF-IDF     | Precision@10: 0.0603

── Fold 2/5 ──
  standard TF-IDF  | Precision@10: 0.0483
  count TF-IDF     | Precision@10: 0.0489

── Fold 3/5 ──
  standard TF-IDF  | Precision@10: 0.0594
  count TF-IDF     | Precision@10: 0.0600

── Fold 4/5 ──
  standard TF-IDF  | Precision@10: 0.0507
  count TF-IDF     | Precision@10: 0.0514

── Fold 5/5 ──
  standard TF-IDF  | Precision@10: 0.0600
  count TF-IDF     | Precision@10: 0.0604

=== Average Precision@10 across 5 folds ===
standard TF-IDF  | Avg Precision@10: 0.0556
count TF-IDF     | Avg Precision@10: 0.0562


In [ ]:
from scipy.sparse import csr_matrix, diags, bmat

# Precompute interaction counts
interaction_counts = interactions.groupby(['user_id', 'book_id']).size().reset_index(name='count')
count_dict = {(row['user_id'], row['book_id']): row['count']
              for _, row in interaction_counts.iterrows()}

# Step 1: Full weighted matrix
full_matrix = create_weighted_matrix(interactions, n_users, n_items, decay=0.03)

# Step 2: CF hybrid
user_sim_full = cosine_similarity(full_matrix)
user_pred_full = user_based_predict(full_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_matrix.T)
item_pred_full = item_based_predict(full_matrix, item_sim_full)
del item_sim_full

user_log = np.log1p(np.abs(user_pred_full)) * np.sign(user_pred_full)
item_log = np.log1p(np.abs(item_pred_full)) * np.sign(item_pred_full)
del user_pred_full, item_pred_full

cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
del user_log, item_log
print("CF ready!")

# Step 3: Count-weighted TF-IDF content
content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue

    valid_rows = []
    weights = []
    for book_idx in read_indices:
        if book_idx in mapping_author_x2:
            count = count_dict.get((user_idx, book_idx), 1)
            weights.append(np.log1p(count))
            valid_rows.append(mapping_author_x2[book_idx])

    if len(valid_rows) == 0:
        continue

    weights = np.array(weights)
    weights = weights / weights.sum()

    user_profile = np.asarray(
        np.average(tfidf_matrix_author_x2[valid_rows].toarray(),
                  weights=weights, axis=0)
    ).reshape(1, -1)

    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_norm = normalize_matrix(np.log1p(content_pred))
del content_pred
print("Content ready!")

# Step 4: Popularity
full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
pop_norm = normalize_matrix(np.log1p(book_popularity))

# Step 5: Current model
current_model = 0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
del cf_hybrid, content_norm, pop_norm
current_norm = normalize_matrix(current_model)
del current_model
print("Current model ready!")

# Step 6: Graph RWR (alpha=0.7)
R = csr_matrix(full_binary)
zero_users = csr_matrix((n_users, n_users))
zero_items = csr_matrix((n_items, n_items))
adjacency = bmat([[zero_users, R], [R.T, zero_items]], format='csr')
del R, zero_users, zero_items

row_sums = np.array(adjacency.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1
D_inv = diags(1.0 / row_sums)
transition = D_inv.dot(adjacency)
del adjacency, D_inv

alpha = 0.7
n_total = n_users + n_items
graph_scores = np.zeros((n_users, n_items))
batch_size = 200

for batch_start in range(0, n_users, batch_size):
    batch_end = min(batch_start + batch_size, n_users)
    batch_size_actual = batch_end - batch_start

    p = np.zeros((n_total, batch_size_actual))
    for i, user_idx in enumerate(range(batch_start, batch_end)):
        p[user_idx, i] = 1.0

    r = p.copy()
    for _ in range(15):
        r = alpha * transition.dot(r) + (1 - alpha) * p

    graph_scores[batch_start:batch_end, :] = r[n_users:, :].T

    if batch_start % 1000 == 0:
        print(f"  RWR: {batch_start}/{n_users} users...")

graph_norm = normalize_matrix(graph_scores)
del graph_scores, transition
print("Graph ready!")

# Step 7: Final hybrid
hybrid_final = 0.8 * current_norm + 0.2 * graph_norm
del current_norm, graph_norm
print("Final hybrid ready!")

# Step 8: Save submission
K = 10
with open("submission_count_tfidf_graph.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_count_tfidf_graph.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

CF ready!
Content ready!
Current model ready!
  RWR: 0/7838 users...
  RWR: 1000/7838 users...
  RWR: 2000/7838 users...
  RWR: 3000/7838 users...
  RWR: 4000/7838 users...
  RWR: 5000/7838 users...
  RWR: 6000/7838 users...
  RWR: 7000/7838 users...
Graph ready!
Final hybrid ready!
user_id,recommendation
4456,13807 13805 8581 3333 9306 9208 14120 12689 13806 13804
142,1973 1970 1959 1972 1974 1962 1975 806 600 1977
362,4248 4258 4257 4259 4252 2107 4256 1418 4249 4246
1809,8971 6338 11317 8937 5864 7496 5045 8653 9249 4824


# BEEEEEEST SO FAR:

In [ ]:
K_FOLDS = 5

approaches = ['current', 'count_cf', 'count_graph', 'count_all']
scores_per_approach = {a: [] for a in approaches}

# Precompute interaction counts
interaction_counts = interactions.groupby(['user_id', 'book_id']).size().reset_index(name='count')
count_dict = {(row['user_id'], row['book_id']): row['count']
              for _, row in interaction_counts.iterrows()}

def create_count_weighted_matrix(data, n_users, n_items):
    data_matrix = np.zeros((n_users, n_items))
    counts = data.groupby(['user_id', 'book_id']).size().reset_index(name='count')
    for _, row in counts.iterrows():
        data_matrix[int(row['user_id']), int(row['book_id'])] = np.log1p(row['count'])
    return data_matrix

def build_graph(matrix, n_users, n_items, alpha=0.7, n_iter=15):
    from scipy.sparse import csr_matrix, diags, bmat
    R = csr_matrix(matrix)
    zero_users = csr_matrix((n_users, n_users))
    zero_items = csr_matrix((n_items, n_items))
    adjacency = bmat([[zero_users, R], [R.T, zero_items]], format='csr')
    del R, zero_users, zero_items

    row_sums = np.array(adjacency.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1
    from scipy.sparse import diags
    D_inv = diags(1.0 / row_sums)
    transition = D_inv.dot(adjacency)
    del adjacency, D_inv

    n_total = n_users + n_items
    graph_scores = np.zeros((n_users, n_items))
    batch_size = 200

    for batch_start in range(0, n_users, batch_size):
        batch_end = min(batch_start + batch_size, n_users)
        batch_size_actual = batch_end - batch_start

        p = np.zeros((n_total, batch_size_actual))
        for i, user_idx in enumerate(range(batch_start, batch_end)):
            p[user_idx, i] = 1.0

        r = p.copy()
        for _ in range(n_iter):
            r = alpha * transition.dot(r) + (1 - alpha) * p

        graph_scores[batch_start:batch_end, :] = r[n_users:, :].T

    del transition
    return graph_scores

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # Build matrices
    train_temporal = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)
    train_count = create_count_weighted_matrix(train_data, n_users, n_items)
    train_binary = create_data_matrix(train_data, n_users, n_items)

    # Count-weighted TF-IDF content (same for all approaches)
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        valid_rows = []
        weights = []
        for book_idx in train_books:
            if book_idx in mapping_author_x2:
                count = count_dict.get((user_idx, book_idx), 1)
                weights.append(np.log1p(count))
                valid_rows.append(mapping_author_x2[book_idx])
        if len(valid_rows) == 0:
            continue
        weights = np.array(weights) / sum(weights)
        user_profile = np.asarray(
            np.average(tfidf_matrix_author_x2[valid_rows].toarray(),
                      weights=weights, axis=0)
        ).reshape(1, -1)
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(np.log1p(content_pred))
    del content_pred

    # Build graphs
    print("  Building binary graph...")
    graph_binary = normalize_matrix(build_graph(train_binary, n_users, n_items))
    print("  Building count graph...")
    graph_count = normalize_matrix(build_graph(train_count, n_users, n_items))

    for approach in approaches:
        # Choose CF matrix
        if approach in ['current', 'count_graph']:
            train_matrix = train_temporal
        else:
            train_matrix = train_count

        # CF predictions
        user_sim = cosine_similarity(train_matrix)
        user_pred = user_based_predict(train_matrix, user_sim)
        del user_sim

        item_sim = cosine_similarity(train_matrix.T)
        item_pred = item_based_predict(train_matrix, item_sim)
        del item_sim

        user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
        item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
        del user_pred, item_pred

        cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
        del user_log, item_log

        book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
        pop_norm = normalize_matrix(np.log1p(book_popularity))

        current_model = normalize_matrix(
            0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
        )
        del cf_hybrid, pop_norm

        # Choose graph
        if approach in ['current', 'count_cf']:
            graph = graph_binary
        else:
            graph = graph_count

        hybrid = 0.8 * current_model + 0.2 * graph
        precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
        scores_per_approach[approach].append(precision)
        print(f"  {approach:15s} | Precision@10: {precision:.4f}")

        del current_model, hybrid

    del train_temporal, train_count, train_binary
    del graph_binary, graph_count, content_norm

# Results
print("\n=== Average Precision@10 across 5 folds ===")
best_approach, best_score = None, 0
for a in approaches:
    avg = np.mean(scores_per_approach[a])
    print(f"{a:15s} | Avg Precision@10: {avg:.4f}")
    if avg > best_score:
        best_score = avg
        best_approach = a

print(f"\n✓ Best: {best_approach} → Avg Precision@10: {best_score:.4f}")


── Fold 1/5 ──
  Building binary graph...
  Building count graph...
  current         | Precision@10: 0.0603
  count_cf        | Precision@10: 0.0610
  count_graph     | Precision@10: 0.0603
  count_all       | Precision@10: 0.0611

── Fold 2/5 ──
  Building binary graph...
  Building count graph...
  current         | Precision@10: 0.0489
  count_cf        | Precision@10: 0.0500
  count_graph     | Precision@10: 0.0489
  count_all       | Precision@10: 0.0500

── Fold 3/5 ──
  Building binary graph...
  Building count graph...
  current         | Precision@10: 0.0597
  count_cf        | Precision@10: 0.0601
  count_graph     | Precision@10: 0.0598
  count_all       | Precision@10: 0.0602

── Fold 4/5 ──
  Building binary graph...
  Building count graph...
  current         | Precision@10: 0.0514
  count_cf        | Precision@10: 0.0508
  count_graph     | Precision@10: 0.0513
  count_all       | Precision@10: 0.0509

── Fold 5/5 ──
  Building binary graph...
  Building count graph...

# Train on full data and save the file for count_cf (temporal TF-IDF + count CF + binary graph) - NO IMPROVEMENT

In [ ]:
from scipy.sparse import csr_matrix, diags, bmat

# Precompute counts
interaction_counts = interactions.groupby(['user_id', 'book_id']).size().reset_index(name='count')
count_dict = {(row['user_id'], row['book_id']): row['count']
              for _, row in interaction_counts.iterrows()}

# Step 1: Count CF matrix
full_count_matrix = create_count_weighted_matrix(interactions, n_users, n_items)

user_sim_full = cosine_similarity(full_count_matrix)
user_pred_full = user_based_predict(full_count_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_count_matrix.T)
item_pred_full = item_based_predict(full_count_matrix, item_sim_full)
del item_sim_full

user_log = np.log1p(np.abs(user_pred_full)) * np.sign(user_pred_full)
item_log = np.log1p(np.abs(item_pred_full)) * np.sign(item_pred_full)
del user_pred_full, item_pred_full

cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
del user_log, item_log
print("CF ready!")

# Step 2: Count-weighted TF-IDF
content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_count_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue
    valid_rows = []
    weights = []
    for book_idx in read_indices:
        if book_idx in mapping_author_x2:
            count = count_dict.get((user_idx, book_idx), 1)
            weights.append(np.log1p(count))
            valid_rows.append(mapping_author_x2[book_idx])
    if len(valid_rows) == 0:
        continue
    weights = np.array(weights) / sum(weights)
    user_profile = np.asarray(
        np.average(tfidf_matrix_author_x2[valid_rows].toarray(),
                  weights=weights, axis=0)
    ).reshape(1, -1)
    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_norm = normalize_matrix(np.log1p(content_pred))
del content_pred
print("Content ready!")

# Step 3: Popularity
full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
pop_norm = normalize_matrix(np.log1p(book_popularity))

current_model = normalize_matrix(0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm)
del cf_hybrid, content_norm, pop_norm
print("Current model ready!")

# Step 4: Binary graph
R = csr_matrix(full_binary)
zero_users = csr_matrix((n_users, n_users))
zero_items = csr_matrix((n_items, n_items))
adjacency = bmat([[zero_users, R], [R.T, zero_items]], format='csr')
del R, zero_users, zero_items

row_sums = np.array(adjacency.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1
D_inv = diags(1.0 / row_sums)
transition = D_inv.dot(adjacency)
del adjacency, D_inv

alpha = 0.7
n_total = n_users + n_items
graph_scores = np.zeros((n_users, n_items))

for batch_start in range(0, n_users, 200):
    batch_end = min(batch_start + 200, n_users)
    p = np.zeros((n_total, batch_end - batch_start))
    for i, user_idx in enumerate(range(batch_start, batch_end)):
        p[user_idx, i] = 1.0
    r = p.copy()
    for _ in range(15):
        r = alpha * transition.dot(r) + (1 - alpha) * p
    graph_scores[batch_start:batch_end, :] = r[n_users:, :].T

graph_norm = normalize_matrix(graph_scores)
del graph_scores, transition
print("Graph ready!")

# Step 5: Final hybrid
hybrid_final = 0.8 * current_model + 0.2 * graph_norm
del current_model, graph_norm

# Save
K = 10
with open("submission_count_cf.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_count_cf.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

CF ready!
Content ready!
Current model ready!
Graph ready!
user_id,recommendation
4456,13807 8581 13805 12689 9306 14120 9208 3333 14430 8588
142,1568 1970 1974 1972 1973 1962 600 806 1975 1959
362,4248 4212 858 4236 4211 4219 4196 4207 4246 1418
1809,11317 8971 6338 7496 5864 5045 8937 8653 9249 4236


# Train on full data and save the file for count_all - NO IMPROVEMENT

In [ ]:
# Вместо full_count_matrix используем temporal для графа
# а CF строим на count матрице — это уже count_cf выше

# Для count_all — CF на count, граф тоже на count матрице:
R = csr_matrix(full_count_matrix)  # ← count вместо binary

from scipy.sparse import csr_matrix, diags, bmat

# Precompute counts
interaction_counts = interactions.groupby(['user_id', 'book_id']).size().reset_index(name='count')
count_dict = {(row['user_id'], row['book_id']): row['count']
              for _, row in interaction_counts.iterrows()}

# Step 1: Count CF matrix
full_count_matrix = create_count_weighted_matrix(interactions, n_users, n_items)

user_sim_full = cosine_similarity(full_count_matrix)
user_pred_full = user_based_predict(full_count_matrix, user_sim_full)
del user_sim_full

item_sim_full = cosine_similarity(full_count_matrix.T)
item_pred_full = item_based_predict(full_count_matrix, item_sim_full)
del item_sim_full

user_log = np.log1p(np.abs(user_pred_full)) * np.sign(user_pred_full)
item_log = np.log1p(np.abs(item_pred_full)) * np.sign(item_pred_full)
del user_pred_full, item_pred_full

cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
del user_log, item_log
print("CF ready!")

# Step 2: Count-weighted TF-IDF
content_pred = np.zeros((n_users, n_items))
for user_idx in range(n_users):
    read_indices = np.where(full_count_matrix[user_idx, :] > 0)[0]
    if len(read_indices) == 0:
        continue
    valid_rows = []
    weights = []
    for book_idx in read_indices:
        if book_idx in mapping_author_x2:
            count = count_dict.get((user_idx, book_idx), 1)
            weights.append(np.log1p(count))
            valid_rows.append(mapping_author_x2[book_idx])
    if len(valid_rows) == 0:
        continue
    weights = np.array(weights) / sum(weights)
    user_profile = np.asarray(
        np.average(tfidf_matrix_author_x2[valid_rows].toarray(),
                  weights=weights, axis=0)
    ).reshape(1, -1)
    scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
    for idx, orig_id in enumerate(books['i']):
        if orig_id in book_id_map:
            content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

content_norm = normalize_matrix(np.log1p(content_pred))
del content_pred
print("Content ready!")

# Step 3: Popularity
full_binary = create_data_matrix(interactions, n_users, n_items)
book_popularity = np.array(full_binary.sum(axis=0)).flatten()
pop_norm = normalize_matrix(np.log1p(book_popularity))

current_model = normalize_matrix(0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm)
del cf_hybrid, content_norm, pop_norm
print("Current model ready!")

# Step 4: Binary graph
R = csr_matrix(full_binary)
zero_users = csr_matrix((n_users, n_users))
zero_items = csr_matrix((n_items, n_items))
adjacency = bmat([[zero_users, R], [R.T, zero_items]], format='csr')
del R, zero_users, zero_items

row_sums = np.array(adjacency.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1
D_inv = diags(1.0 / row_sums)
transition = D_inv.dot(adjacency)
del adjacency, D_inv

alpha = 0.7
n_total = n_users + n_items
graph_scores = np.zeros((n_users, n_items))

for batch_start in range(0, n_users, 200):
    batch_end = min(batch_start + 200, n_users)
    p = np.zeros((n_total, batch_end - batch_start))
    for i, user_idx in enumerate(range(batch_start, batch_end)):
        p[user_idx, i] = 1.0
    r = p.copy()
    for _ in range(15):
        r = alpha * transition.dot(r) + (1 - alpha) * p
    graph_scores[batch_start:batch_end, :] = r[n_users:, :].T

graph_norm = normalize_matrix(graph_scores)
del graph_scores, transition
print("Graph ready!")

# Step 5: Final hybrid
hybrid_final = 0.8 * current_model + 0.2 * graph_norm
del current_model, graph_norm

# Save
K = 10
with open("submission_count_all.csv", "w") as f:
    f.write("user_id,recommendation\n")
    for user_idx in range(hybrid_final.shape[0]):
        scores = hybrid_final[user_idx, :].copy()
        top_k_idx = np.argsort(scores)[::-1][:K]
        original_book_ids = [str(book_id_inverse_map[idx]) for idx in top_k_idx]
        original_user_id = user_id_inverse_map[user_idx]
        f.write(f"{original_user_id},{' '.join(original_book_ids)}\n")

with open("submission_count_all.csv", "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: break

CF ready!
Content ready!
Current model ready!
Graph ready!
user_id,recommendation
4456,13807 8581 13805 12689 9306 14120 9208 3333 14430 8588
142,1568 1970 1974 1972 1973 1962 600 806 1975 1959
362,4248 4212 858 4236 4211 4219 4196 4207 4246 1418
1809,11317 8971 6338 7496 5864 5045 8937 8653 9249 4236


# Idea 11: Let's combine the best 2 recommendations (2482 users get different recommendations in these 2 files) - NO IMPROVEMENT


In [ ]:
import pandas as pd
import numpy as np

sub1 = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/submission3_log.csv")        # 0.1724
sub2 = pd.read_csv("https://raw.githubusercontent.com/MiraFedo/Machine-Learning-/main/submission8_count_tfidf_graph.csv")  # 0.1739


# Convert rank-based scores
# Position 1 → score 10, position 2 → score 9, ... position 10 → score 1
def to_score_matrix(sub, all_user_ids, all_book_ids):
    user_id_map = {uid: i for i, uid in enumerate(all_user_ids)}
    book_id_map = {bid: i for i, bid in enumerate(all_book_ids)}
    matrix = np.zeros((len(all_user_ids), len(all_book_ids)))

    for _, row in sub.iterrows():
        user_idx = user_id_map.get(row['user_id'])
        if user_idx is None:
            continue
        books = [int(b) for b in str(row['recommendation']).split()]
        for rank, book_id in enumerate(books):
            book_idx = book_id_map.get(book_id)
            if book_idx is not None:
                matrix[user_idx, book_idx] = 10 - rank  # rank 0 → score 10
    return matrix

all_user_ids = sorted(sub1['user_id'].unique())
all_book_ids = sorted(set(
    int(b) for rec in sub1['recommendation'] for b in rec.split()
) | set(
    int(b) for rec in sub2['recommendation'] for b in rec.split()
))

print(f"Users: {len(all_user_ids)}, Books: {len(all_book_ids)}")

m1 = to_score_matrix(sub1, all_user_ids, all_book_ids)
m2 = to_score_matrix(sub2, all_user_ids, all_book_ids)

# Try different weights
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ensemble = w * m2 + (1 - w) * m1  # w for best model (sub2)

    # Generate recommendations
    rows = []
    user_id_map = {i: uid for i, uid in enumerate(all_user_ids)}
    book_id_map_inv = {i: bid for i, bid in enumerate(all_book_ids)}

    for user_idx in range(len(all_user_ids)):
        scores = ensemble[user_idx, :]
        top_k_idx = np.argsort(scores)[::-1][:10]
        top_k_books = [str(book_id_map_inv[idx]) for idx in top_k_idx if scores[idx] > 0]
        rows.append({
            'user_id': user_id_map[user_idx],
            'recommendation': ' '.join(top_k_books[:10])
        })

    df = pd.DataFrame(rows)
    df.to_csv(f"ensemble_w{int(w*10)}.csv", index=False)
    print(f"Saved ensemble_w{int(w*10)}.csv (w={w} for best model)")

print("Done!")

Users: 7838, Books: 14774
Saved ensemble_w3.csv (w=0.3 for best model)
Saved ensemble_w4.csv (w=0.4 for best model)
Saved ensemble_w5.csv (w=0.5 for best model)
Saved ensemble_w6.csv (w=0.6 for best model)
Saved ensemble_w7.csv (w=0.7 for best model)
Done!


# Look at the data

In [ ]:
# Check if books with similar IDs have similar subjects
books_sorted = books.sort_values('i')
print(books_sorted[['i', 'Title', 'Subjects']].head(20).to_string())

# Check correlation between ID proximity and co-reading
interactions_merged = interactions.merge(interactions, on='user_id', suffixes=('_1', '_2'))
interactions_merged['id_diff'] = abs(interactions_merged['book_id_1'] - interactions_merged['book_id_2'])
print(interactions_merged.groupby('id_diff')['user_id'].count().head(20))

     i                                                                                                                  Title                                                                                                                                                                                                            Subjects
0    0                                                                Classification décimale universelle : édition abrégée /                                                       Classification décimale universelle; Indexation (documentation); classification décimale universelle--[tables]; Book Classification; Reference Books; Outline
1    1                           Les interactions dans l'enseignement des langues : agir professoral et pratiques de classe /                                                                                                                                      didactique--langue étrangère - enseignement; didactique--langue -

In [ ]:
K_FOLDS = 5
scores_without_proximity = []
scores_with_proximity = []

# Precompute book IDs array for proximity calculation
book_ids_array = np.array([book_id_inverse_map[i] for i in range(n_items)])

interaction_counts = interactions.groupby(['user_id', 'book_id']).size().reset_index(name='count')
count_dict = {(row['user_id'], row['book_id']): row['count']
              for _, row in interaction_counts.iterrows()}

interactions_sorted = interactions.sort_values(["user_id", "timestamp"]).copy()
interactions_sorted["fold"] = interactions_sorted.groupby("user_id")["timestamp"].transform(
    lambda x: pd.qcut(x.rank(method='first'), K_FOLDS, labels=False)
)

# Try different sigma values for proximity
sigmas = [5, 10, 20, 50]
scores_per_sigma = {s: [] for s in sigmas}
scores_per_sigma['no_proximity'] = []

for fold in range(K_FOLDS):
    print(f"\n── Fold {fold+1}/{K_FOLDS} ──")

    train_data = interactions_sorted[interactions_sorted["fold"] != fold]
    test_data = interactions_sorted[interactions_sorted["fold"] == fold]

    train_matrix = create_weighted_matrix(train_data, n_users, n_items, decay=0.03)

    test_matrix = np.zeros((n_users, n_items))
    test_matrix[test_data["user_id"].values, test_data["book_id"].values] = 1

    # CF
    user_sim = cosine_similarity(train_matrix)
    user_pred = user_based_predict(train_matrix, user_sim)
    del user_sim

    item_sim = cosine_similarity(train_matrix.T)
    item_pred = item_based_predict(train_matrix, item_sim)
    del item_sim

    user_log = np.log1p(np.abs(user_pred)) * np.sign(user_pred)
    item_log = np.log1p(np.abs(item_pred)) * np.sign(item_pred)
    del user_pred, item_pred

    cf_hybrid = 0.45 * normalize_matrix(user_log) + 0.55 * normalize_matrix(item_log)
    del user_log, item_log

    # Count-weighted TF-IDF content
    content_pred = np.zeros((n_users, n_items))
    for user_idx in range(n_users):
        train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
        if len(train_books) == 0:
            continue
        valid_rows, weights = [], []
        for book_idx in train_books:
            if book_idx in mapping_author_x2:
                count = count_dict.get((user_idx, book_idx), 1)
                weights.append(np.log1p(count))
                valid_rows.append(mapping_author_x2[book_idx])
        if len(valid_rows) == 0:
            continue
        weights = np.array(weights) / sum(weights)
        user_profile = np.asarray(
            np.average(tfidf_matrix_author_x2[valid_rows].toarray(),
                      weights=weights, axis=0)
        ).reshape(1, -1)
        scores = cosine_similarity(user_profile, tfidf_matrix_author_x2).flatten()
        for idx, orig_id in enumerate(books['i']):
            if orig_id in book_id_map:
                content_pred[user_idx, book_id_map[orig_id]] = scores[idx]

    content_norm = normalize_matrix(np.log1p(content_pred))
    del content_pred

    # Popularity
    book_popularity = np.array(train_matrix.sum(axis=0)).flatten()
    pop_norm = normalize_matrix(np.log1p(book_popularity))

    # Graph
    train_binary = create_data_matrix(train_data, n_users, n_items)
    from scipy.sparse import csr_matrix, diags, bmat
    R = csr_matrix(train_binary)
    zero_users = csr_matrix((n_users, n_users))
    zero_items = csr_matrix((n_items, n_items))
    adjacency = bmat([[zero_users, R], [R.T, zero_items]], format='csr')
    del R, zero_users, zero_items

    row_sums = np.array(adjacency.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1
    D_inv = diags(1.0 / row_sums)
    transition = D_inv.dot(adjacency)
    del adjacency, D_inv

    n_total = n_users + n_items
    graph_scores = np.zeros((n_users, n_items))
    for batch_start in range(0, n_users, 200):
        batch_end = min(batch_start + 200, n_users)
        p = np.zeros((n_total, batch_end - batch_start))
        for i, user_idx in enumerate(range(batch_start, batch_end)):
            p[user_idx, i] = 1.0
        r = p.copy()
        for _ in range(15):
            r = 0.7 * transition.dot(r) + 0.3 * p
        graph_scores[batch_start:batch_end, :] = r[n_users:, :].T

    graph_norm = normalize_matrix(graph_scores)
    del graph_scores, transition

    # Base model without proximity
    base_model = normalize_matrix(
        0.75 * cf_hybrid + 0.20 * content_norm + 0.05 * pop_norm
    )
    hybrid_no_prox = 0.8 * base_model + 0.2 * graph_norm
    precision, _ = precision_recall_at_k(hybrid_no_prox, test_matrix, k=10)
    scores_per_sigma['no_proximity'].append(precision)
    print(f"  no_proximity | Precision@10: {precision:.4f}")
    del hybrid_no_prox

    # ID proximity scores for each sigma
    for sigma in sigmas:
        proximity_matrix = np.zeros((n_users, n_items))
        for user_idx in range(n_users):
            train_books = train_data[train_data['user_id'] == user_idx]['book_id'].values
            if len(train_books) == 0:
                continue
            # Get original book IDs
            orig_book_ids = np.array([book_id_inverse_map[b] for b in train_books
                                      if b in book_id_inverse_map])
            if len(orig_book_ids) == 0:
                continue
            # Gaussian kernel around each book ID
            prox_scores = np.zeros(n_items)
            for orig_id in orig_book_ids:
                prox_scores += np.exp(
                    -((book_ids_array - orig_id) ** 2) / (2 * sigma ** 2)
                )
            # Zero out books user already read
            prox_scores[train_books[train_books < n_items]] = 0
            proximity_matrix[user_idx, :] = prox_scores

        prox_norm = normalize_matrix(proximity_matrix)
        del proximity_matrix

        # Add proximity to model
        hybrid_prox = 0.8 * base_model + 0.2 * graph_norm
        # Replace small part with proximity
        for prox_w in [0.05, 0.10]:
            hybrid = (1 - prox_w) * hybrid_prox + prox_w * prox_norm
            precision, _ = precision_recall_at_k(hybrid, test_matrix, k=10)
            print(f"  sigma={sigma:3d} prox_w={prox_w:.2f} | Precision@10: {precision:.4f}")
            del hybrid

        del prox_norm, hybrid_prox

    del cf_hybrid, content_norm, pop_norm, graph_norm, base_model, train_binary

# Results
print("\n=== Average Precision@10 across 5 folds ===")
print(f"no_proximity | Avg: {np.mean(scores_per_sigma['no_proximity']):.4f}")


── Fold 1/5 ──
  no_proximity | Precision@10: 0.0603
  sigma=  5 prox_w=0.05 | Precision@10: 0.0603
  sigma=  5 prox_w=0.10 | Precision@10: 0.0604
  sigma= 10 prox_w=0.05 | Precision@10: 0.0603
  sigma= 10 prox_w=0.10 | Precision@10: 0.0604
  sigma= 20 prox_w=0.05 | Precision@10: 0.0603
  sigma= 20 prox_w=0.10 | Precision@10: 0.0604
  sigma= 50 prox_w=0.05 | Precision@10: 0.0603
  sigma= 50 prox_w=0.10 | Precision@10: 0.0603

── Fold 2/5 ──
  no_proximity | Precision@10: 0.0489
  sigma=  5 prox_w=0.05 | Precision@10: 0.0490
  sigma=  5 prox_w=0.10 | Precision@10: 0.0491
  sigma= 10 prox_w=0.05 | Precision@10: 0.0490
  sigma= 10 prox_w=0.10 | Precision@10: 0.0491
  sigma= 20 prox_w=0.05 | Precision@10: 0.0490
  sigma= 20 prox_w=0.10 | Precision@10: 0.0490
  sigma= 50 prox_w=0.05 | Precision@10: 0.0490
  sigma= 50 prox_w=0.10 | Precision@10: 0.0490

── Fold 3/5 ──
  no_proximity | Precision@10: 0.0597
  sigma=  5 prox_w=0.05 | Precision@10: 0.0598
  sigma=  5 prox_w=0.10 | Precision@10: